### ЗАДАЧА: Реестр абонементов фитнес-клуба

Администратор фитнес-клуба получает строки с данными об абонементах.
Нужно собрать удобную модель, которая позволит:
- загрузить клиентов в единый реестр,
- посмотреть только активные абонементы,
- отфильтровать клиентов по тарифу,
- посчитать суммарное число оставшихся посещений,
- понять, как меняется реестр после активации и списания посещения.

В данных есть абонементы с разными статусами и остатком посещений,
поэтому важно корректно валидировать тариф, статус и изменение состояния объекта.


In [2]:
# rows: sub_id|client_name|plan|visits_left|status
rows = [
    'SB-100|Alice|standard|8|active',
    'SB-101|Bob|premium|12|frozen',
    'SB-102|Charlie|family|0|expired',
    'SB-103|Diana|standard|5|active',
]


class Subscription:
    allowed_plans = {'standard', 'premium', 'family'}
    allowed_statuses = {'active', 'frozen', 'expired'}

    def __init__(self, sub_id, client_name, plan, visits_left, status):
        # TODO: сохранить sub_id, client_name, plan
        # TODO: visits_left хранить через self._visits_left
        # TODO: значение visits_left пропустить через property/setter
        # TODO: проверить plan и status, иначе raise ValueError
        self.sub_id = sub_id
        self.client_name = client_name
        if plan not in self.allowed_plans:
            raise ValueError(f"Plan '{plan}' is not allowed.Allowed: {self.allowed_plans}")
        self.plan = plan
        self.visits_left = visits_left
        if status not in self.allowed_statuses:
            raise ValueError(f"Status '{status}' is not allowed. Allowed: {self.allowed_statuses}")
        self.status = status

    @property
    def visits_left(self):
        # TODO: вернуть текущее число посещений
           return self._visits_left

    @visits_left.setter
    def visits_left(self, value):
        # TODO: привести value к int
        # TODO: если value < 0 -> raise ValueError('Visits must be >= 0')
        # TODO: сохранить результат в self._visits_left
        value = int(value)
        if value < 0:
            raise ValueError('Visits must be >= 0')
        self._visits_left = value
       

    def use_visit(self):
        # TODO: если статус не 'active' -> raise ValueError
        # TODO: если visits_left == 0 -> raise ValueError
        # TODO: уменьшить visits_left на 1
        # TODO: если после списания visits_left == 0, перевести статус в 'expired'
        if self.status != 'active':
            raise ValueError(f"Cannot use visit: subscription is {self.status}, must be 'active'")
        if self.visits_left == 0:
            raise ValueError("Cannot use visit: no visits left")
        self.visits_left -= 1
        if self.visits_left == 0:
            self.status = 'expired'

    def freeze(self):
        # TODO: если статус 'expired' -> raise ValueError
        # TODO: перевести абонемент в 'frozen'
        if self.status == 'expired':
            raise ValueError("Cannot freeze: subscription is already expired")
        self.status = 'frozen'

    def activate(self):
        # TODO: если visits_left == 0 -> raise ValueError
        # TODO: перевести абонемент в 'active'
        if self.visits_left == 0:
            raise ValueError("Cannot activate: no visits left")
        self.status = 'active'

    @classmethod
    def from_row(cls, row):
        # TODO: split по '|'
        # TODO: ожидать 5 частей: sub_id, client_name, plan, visits_left, status
        # TODO: вернуть Subscription(...)
        parts = row.split('|')
        if len(parts) != 5:
            raise ValueError(f"Invalid row format: expected 5 parts, got {len(parts)}")
        sub_id, client_name, plan, visits_left, status = parts
        return cls(sub_id, client_name, plan, visits_left, status)

    def __repr__(self):
        # TODO: вернуть строку вида Subscription(sub_id='...', client_name='...', status='...')
        return f"Subscription(sub_id='{self.sub_id}', client_name='{self.client_name}', status='{self.status}')"


class SubscriptionRegistry:
    def __init__(self):
        self.items = []

    def add(self, subscription):
        # TODO: добавить subscription в self.items
        self.items.append(subscription)

    def load(self, rows):
        # TODO: для каждой строки создать Subscription.from_row(row)
        # TODO: добавить объект в реестр через add(...)
        for row in rows:
            subscription = Subscription.from_row(row)
            self.add(subscription)

    def active_subscriptions(self):
        # TODO: вернуть список абонементов со статусом 'active'
        return [sub for sub in self.items if sub.status == 'active']

    def by_plan(self, plan):
        # TODO: вернуть список абонементов нужного тарифа
        return [sub for sub in self.items if sub.plan == plan]

    def total_visits_left(self):
        # TODO: вернуть суммарное число оставшихся посещений
        return sum(sub.visits_left for sub in self.items)

    def status_summary(self):
        # TODO: собрать dict вида status -> count
        summary = {}
        for sub in self.items:
            status = sub.status
            summary[status] = summary.get(status, 0) + 1
        return summary

    def find(self, sub_id):
        # TODO: вернуть абонемент по sub_id или None
        for sub in self.items:
            if sub.sub_id == sub_id:
                return sub
        return None


registry = SubscriptionRegistry()

# TODO: загрузить rows в registry
registry.load(rows)

# TODO: вывести все абонементы
print("Все абонементы:")
for sub in registry.items:
    print(sub)
    
# TODO: вывести active_subscriptions()
print("Активные абонементы:")
print(registry.active_subscriptions())

# TODO: вывести by_plan('standard')
print("Абонементы тарифа 'standard':")
print(registry.by_plan('standard'))

# TODO: вывести total_visits_left()
print("Суммарное число оставшихся посещений:")
print(registry.total_visits_left())

# TODO: вывести status_summary()
print("Сводка по статусам:")
print(registry.status_summary())

# TODO: найти абонемент 'SB-101', активировать его и вывести status_summary()
sub_sb101 = registry.find('SB-101')
if sub_sb101:
    sub_sb101.activate()
    print("После активации SB-101 сводка по статусам:")
    print(registry.status_summary())
    
# TODO: найти абонемент 'SB-100', списать одно посещение и вывести объект
sub_sb100 = registry.find('SB-100')
if sub_sb100:
    sub_sb100.use_visit()
    print("После списания посещения с SB-100:")
    print(sub_sb100)


Все абонементы:
Subscription(sub_id='SB-100', client_name='Alice', status='active')
Subscription(sub_id='SB-101', client_name='Bob', status='frozen')
Subscription(sub_id='SB-102', client_name='Charlie', status='expired')
Subscription(sub_id='SB-103', client_name='Diana', status='active')
Активные абонементы:
[Subscription(sub_id='SB-100', client_name='Alice', status='active'), Subscription(sub_id='SB-103', client_name='Diana', status='active')]
Абонементы тарифа 'standard':
[Subscription(sub_id='SB-100', client_name='Alice', status='active'), Subscription(sub_id='SB-103', client_name='Diana', status='active')]
Суммарное число оставшихся посещений:
25
Сводка по статусам:
{'active': 2, 'frozen': 1, 'expired': 1}
После активации SB-101 сводка по статусам:
{'active': 3, 'expired': 1}
После списания посещения с SB-100:
Subscription(sub_id='SB-100', client_name='Alice', status='active')
